In [0]:
# Data Quality Summary

from pyspark.sql import Row

CATALOG = "insurance_lakehouse"

datasets = [
    {
        "dataset": "customers",
        "bronze_table": f"{CATALOG}.bronze.bronze_customers",
        "silver_table": f"{CATALOG}.silver.silver_customers",
        "quarantine_table": f"{CATALOG}.quarantine.quarantine_invalid_customers"
    },
    {
        "dataset": "policies",
        "bronze_table": f"{CATALOG}.bronze.bronze_policies",
        "silver_table": f"{CATALOG}.silver.silver_policies",
        "quarantine_table": f"{CATALOG}.quarantine.quarantine_invalid_policies"
    },
    {
        "dataset": "claims",
        "bronze_table": f"{CATALOG}.bronze.bronze_claims",
        "silver_table": f"{CATALOG}.silver.silver_claims",
        "quarantine_table": f"{CATALOG}.quarantine.quarantine_invalid_claims"
    },
    {
        "dataset": "payments",
        "bronze_table": f"{CATALOG}.bronze.bronze_payments",
        "silver_table": f"{CATALOG}.silver.silver_payments",
        "quarantine_table": f"{CATALOG}.quarantine.quarantine_invalid_payments"
    },
    {
        "dataset": "agents",
        "bronze_table": f"{CATALOG}.bronze.bronze_agents",
        "silver_table": f"{CATALOG}.silver.silver_agents",
        "quarantine_table": None
    },
    {
        "dataset": "fraud_indicators",
        "bronze_table": f"{CATALOG}.bronze.bronze_fraud_indicators",
        "silver_table": f"{CATALOG}.silver.silver_fraud_indicators",
        "quarantine_table": f"{CATALOG}.quarantine.quarantine_invalid_fraud_indicators"
    }
]

rows = []

for item in datasets:
    bronze_rows = spark.table(item["bronze_table"]).count()
    silver_rows = spark.table(item["silver_table"]).count()
    
    if item["quarantine_table"]:
        quarantine_rows = spark.table(item["quarantine_table"]).count()
    else:
        quarantine_rows = 0
    
    status = "PASS" if bronze_rows == silver_rows + quarantine_rows else "CHECK"
    
    rows.append(
        Row(
            dataset=item["dataset"],
            bronze_rows=bronze_rows,
            silver_rows=silver_rows,
            quarantine_rows=quarantine_rows,
            status=status
        )
    )

quality_summary_df = spark.createDataFrame(rows)

display(quality_summary_df)